In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pag-synthesize ng mga unitary na operasyon

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
```
</details>
Ang isang unitary na operasyon ay naglalarawan ng isang pagbabago sa quantum system na nagpapanatili ng norm.
Para sa $n$ na mga qubit, ang pagbabagong ito ay inilalarawan ng isang $2^n \times 2^n$ na dimensyon, kumplikadong matrix $U$ na ang adjoint ay katumbas ng inverse, iyon ay $U^\dagger U = \mathbb{1}$.

Ang pag-synthesize ng mga tiyak na unitary na operasyon sa isang hanay ng mga quantum gate ay isang pangunahing gawain na ginagamit, halimbawa, sa disenyo at aplikasyon ng mga quantum algorithm o sa pag-compile ng mga quantum circuit.

Bagama't posible ang mahusay na synthesis para sa ilang uri ng mga unitary — tulad ng mga binubuo ng mga Clifford gate o may tensor product na istraktura — karamihan sa mga unitary ay hindi napapabilang sa mga kategoryang ito.
Para sa mga pangkalahatang unitary matrix, ang synthesis ay isang kumplikadong gawain na ang computational cost ay tumataas nang exponential habang dumadami ang bilang ng mga qubit.
Kaya naman, kung alam mo ang isang mahusay na decomposition para sa unitary na gusto mong ipatupad, malamang na mas maganda pa ito kaysa sa isang pangkalahatang synthesis.

> **Note:** Kung walang available na decomposition, nagbibigay ang Qiskit SDK ng mga tool para mahanap ito.
>     Gayunpaman, tandaan na sa pangkalahatan ay gumagawa ito ng malalim na mga circuit na maaaring hindi angkop para patakbuhin sa mga maingay na quantum computer.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## Re-synthesis para sa pag-optimize ng circuit
Minsan ay kapaki-pakinabang ang muling pag-synthesize ng mahabang serye ng mga single- at two-qubit gate, kung mababawasan ang haba. Halimbawa, ang sumusunod na circuit ay gumagamit ng tatlong two-qubit gate.

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Output of the previous code cell](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

Gayunpaman, pagkatapos ng muling pag-synthesize gamit ang sumusunod na code, isang CX gate na lang ang kailangan nito. (Dito ay ginagamit namin ang `QuantumCircuit.decompose()` na paraan para mas mavisuwalisaw ang mga gate na ginamit sa muling pag-synthesize ng unitary.)